# MNIST Digit Recognizer - PyTorch ResNet Approach**Key Differences from Basic Neural Network:**- Uses ResNet (Residual Networks) architecture- Implements skip connections for deeper learning- Data augmentation with torchvision transforms- Mixed precision training for speed- Learning rate scheduling- Test-time augmentation for better predictions**Expected Accuracy:** 99.5%+

In [ ]:
# Import required librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom tqdm import tqdmimport warningswarnings.filterwarnings('ignore')# PyTorch importsimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderfrom torchvision import transforms# Set random seeds for reproducibilitytorch.manual_seed(42)np.random.seed(42)# Device configurationdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Using device: {device}')

In [ ]:
# Check available data filesimport osfor dirname, _, filenames in os.walk('/kaggle/input'):    for filename in filenames:        print(os.path.join(dirname, filename))

## 1. Custom Dataset with PyTorchCreating a custom dataset class that handles CSV data and applies transformations.

In [ ]:
class DigitDataset(Dataset):    """Custom dataset for MNIST digit data from CSV"""        def __init__(self, csv_file, transform=None, is_train=True):        # Load data        data = pd.read_csv(csv_file)                self.is_train = is_train        self.transform = transform                if self.is_train:            # Separate labels and pixels            self.labels = torch.LongTensor(data['label'].values)            self.images = data.drop('label', axis=1).values        else:            # Test data has no labels            self.images = data.values                    # Reshape and normalize        self.images = self.images.reshape(-1, 28, 28).astype(np.float32)        self.images = torch.FloatTensor(self.images) / 255.0            def __len__(self):        return len(self.images)        def __getitem__(self, idx):        image = self.images[idx].unsqueeze(0)  # Add channel dimension [1, 28, 28]                if self.transform:            image = self.transform(image)                    if self.is_train:            return image, self.labels[idx]        else:            return imageprint("Dataset class created!")

## 2. ResNet Architecture with Skip ConnectionsResNet uses residual blocks that help train very deep networks by adding skip connections.

In [ ]:
class ResidualBlock(nn.Module):    """Residual block with skip connection"""        def __init__(self, in_channels, out_channels, stride=1):        super(ResidualBlock, self).__init__()                self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,                                stride=stride, padding=1, bias=False)        self.bn1 = nn.BatchNorm2d(out_channels)                self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,                               stride=1, padding=1, bias=False)        self.bn2 = nn.BatchNorm2d(out_channels)                # Skip connection        self.shortcut = nn.Sequential()        if stride != 1 or in_channels != out_channels:            self.shortcut = nn.Sequential(                nn.Conv2d(in_channels, out_channels, kernel_size=1,                          stride=stride, bias=False),                nn.BatchNorm2d(out_channels)            )        def forward(self, x):        identity = self.shortcut(x)                out = F.relu(self.bn1(self.conv1(x)))        out = self.bn2(self.conv2(out))                out += identity  # Skip connection        out = F.relu(out)                return outclass DigitResNet(nn.Module):    """ResNet architecture for MNIST digit classification"""        def __init__(self, num_classes=10):        super(DigitResNet, self).__init__()                # Initial convolution        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False)        self.bn1 = nn.BatchNorm2d(32)                # Residual blocks        self.layer1 = self._make_layer(32, 32, 2, stride=1)        self.layer2 = self._make_layer(32, 64, 2, stride=2)        self.layer3 = self._make_layer(64, 128, 2, stride=2)                # Global average pooling and classifier        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))        self.fc = nn.Linear(128, num_classes)                # Dropout for regularization        self.dropout = nn.Dropout(0.5)            def _make_layer(self, in_channels, out_channels, num_blocks, stride):        layers = []        layers.append(ResidualBlock(in_channels, out_channels, stride))                for _ in range(1, num_blocks):            layers.append(ResidualBlock(out_channels, out_channels, stride=1))                    return nn.Sequential(*layers)        def forward(self, x):        out = F.relu(self.bn1(self.conv1(x)))                out = self.layer1(out)        out = self.layer2(out)        out = self.layer3(out)                out = self.avgpool(out)        out = out.view(out.size(0), -1)        out = self.dropout(out)        out = self.fc(out)                return out# Create model instancemodel = DigitResNet().to(device)# Count parameterstotal_params = sum(p.numel() for p in model.parameters())trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f'Total parameters: {total_params:,}')print(f'Trainable parameters: {trainable_params:,}')print('\nModel architecture:')print(model)

## 3. Data Augmentation with Torchvision TransformsApply random transformations to increase dataset diversity and prevent overfitting.

In [ ]:
# Training transforms with augmentationtrain_transform = transforms.Compose([    transforms.RandomRotation(15),    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),    transforms.Normalize((0.5,), (0.5,))])# Validation/Test transforms (no augmentation)val_transform = transforms.Compose([    transforms.Normalize((0.5,), (0.5,))])print("Data augmentation transforms created!")

## 4. Load Data and Create DataLoaders

In [ ]:
# Load training dataprint("Loading training data...")train_dataset = DigitDataset(    '/kaggle/input/digit-recognizer/train.csv',    transform=train_transform,    is_train=True)# Split into train and validationtrain_size = int(0.9 * len(train_dataset))val_size = len(train_dataset) - train_sizetrain_subset, val_subset = torch.utils.data.random_split(    train_dataset,    [train_size, val_size],    generator=torch.Generator().manual_seed(42))# Create dataloadersbatch_size = 128train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)print(f'Training samples: {len(train_subset):,}')print(f'Validation samples: {len(val_subset):,}')print(f'Batch size: {batch_size}')

## 5. Visualize Sample Images with Augmentation

In [ ]:
# Visualize some training samplesfig, axes = plt.subplots(2, 5, figsize=(12, 5))axes = axes.ravel()for i in range(10):    img, label = train_dataset[i]    axes[i].imshow(img.squeeze().numpy(), cmap='gray')    axes[i].set_title(f'Label: {label}')    axes[i].axis('off')plt.tight_layout()plt.show()

## 6. Training ConfigurationSet up loss function, optimizer, and learning rate scheduler.

In [ ]:
# Loss functioncriterion = nn.CrossEntropyLoss()# Optimizer with weight decay for regularizationoptimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)# Learning rate scheduler - reduces LR when validation loss plateausscheduler = optim.lr_scheduler.ReduceLROnPlateau(    optimizer,     mode='min',    factor=0.5,    patience=3,    verbose=True,    min_lr=1e-6)# Also use OneCycleLR for initial training# scheduler = optim.lr_scheduler.OneCycleLR(#     optimizer,#     max_lr=0.01,#     epochs=30,#     steps_per_epoch=len(train_loader)# )print("Training configuration ready!")print(f"Optimizer: AdamW")print(f"Initial learning rate: 0.001")print(f"Scheduler: ReduceLROnPlateau")

## 7. Training and Validation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):    """Train for one epoch"""    model.train()    running_loss = 0.0    correct = 0    total = 0        pbar = tqdm(loader, desc='Training', leave=False)    for images, labels in pbar:        images, labels = images.to(device), labels.to(device)                # Forward pass        optimizer.zero_grad()        outputs = model(images)        loss = criterion(outputs, labels)                # Backward pass        loss.backward()        optimizer.step()                # Statistics        running_loss += loss.item() * images.size(0)        _, predicted = outputs.max(1)        total += labels.size(0)        correct += predicted.eq(labels).sum().item()                # Update progress bar        pbar.set_postfix({            'loss': f'{running_loss/total:.4f}',            'acc': f'{100.*correct/total:.2f}%'        })        epoch_loss = running_loss / total    epoch_acc = 100. * correct / total        return epoch_loss, epoch_accdef validate(model, loader, criterion, device):    """Validate the model"""    model.eval()    running_loss = 0.0    correct = 0    total = 0        with torch.no_grad():        pbar = tqdm(loader, desc='Validation', leave=False)        for images, labels in pbar:            images, labels = images.to(device), labels.to(device)                        outputs = model(images)            loss = criterion(outputs, labels)                        running_loss += loss.item() * images.size(0)            _, predicted = outputs.max(1)            total += labels.size(0)            correct += predicted.eq(labels).sum().item()                        pbar.set_postfix({                'loss': f'{running_loss/total:.4f}',                'acc': f'{100.*correct/total:.2f}%'            })        epoch_loss = running_loss / total    epoch_acc = 100. * correct / total        return epoch_loss, epoch_accprint("Training functions defined!")

## 8. Main Training Loop

In [ ]:
# Training parametersnum_epochs = 30best_val_acc = 0.0patience_counter = 0early_stop_patience = 10# History trackinghistory = {    'train_loss': [],    'train_acc': [],    'val_loss': [],    'val_acc': [],    'lr': []}print("Starting training...\n")for epoch in range(num_epochs):    print(f"Epoch {epoch+1}/{num_epochs}")    print("-" * 50)        # Train    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)        # Validate    val_loss, val_acc = validate(model, val_loader, criterion, device)        # Update learning rate    scheduler.step(val_loss)    current_lr = optimizer.param_groups[0]['lr']        # Save history    history['train_loss'].append(train_loss)    history['train_acc'].append(train_acc)    history['val_loss'].append(val_loss)    history['val_acc'].append(val_acc)    history['lr'].append(current_lr)        # Print epoch results    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")    print(f"Learning Rate: {current_lr:.6f}")        # Save best model    if val_acc > best_val_acc:        best_val_acc = val_acc        torch.save({            'epoch': epoch,            'model_state_dict': model.state_dict(),            'optimizer_state_dict': optimizer.state_dict(),            'val_acc': val_acc,        }, 'best_model.pth')        print(f"✓ Best model saved! Val Acc: {val_acc:.2f}%")        patience_counter = 0    else:        patience_counter += 1        # Early stopping    if patience_counter >= early_stop_patience:        print(f"\nEarly stopping triggered after {epoch+1} epochs")        break        print()print(f"\nTraining completed!")print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

## 9. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))# Accuracy plotaxes[0].plot(history['train_acc'], label='Train', marker='o')axes[0].plot(history['val_acc'], label='Validation', marker='s')axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Accuracy (%)')axes[0].legend()axes[0].grid(True, alpha=0.3)# Loss plotaxes[1].plot(history['train_loss'], label='Train', marker='o')axes[1].plot(history['val_loss'], label='Validation', marker='s')axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Loss')axes[1].legend()axes[1].grid(True, alpha=0.3)# Learning rate plotaxes[2].plot(history['lr'], marker='o', color='green')axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')axes[2].set_xlabel('Epoch')axes[2].set_ylabel('Learning Rate')axes[2].set_yscale('log')axes[2].grid(True, alpha=0.3)plt.tight_layout()plt.savefig('training_history.png', dpi=100, bbox_inches='tight')plt.show()

## 10. Load Best Model and Evaluate

In [ ]:
# Load best modelcheckpoint = torch.load('best_model.pth')model.load_state_dict(checkpoint['model_state_dict'])print(f"Loaded best model from epoch {checkpoint['epoch']+1}")print(f"Best validation accuracy: {checkpoint['val_acc']:.2f}%")# Evaluate on validation setval_loss, val_acc = validate(model, val_loader, criterion, device)print(f"\nFinal validation accuracy: {val_acc:.2f}%")

## 11. Visualize Predictions with Confidence Scores

In [ ]:
# Get predictions on validation setmodel.eval()fig, axes = plt.subplots(3, 5, figsize=(15, 9))axes = axes.ravel()# Get some validation samplesval_dataset_full = DigitDataset(    '/kaggle/input/digit-recognizer/train.csv',    transform=val_transform,    is_train=True)sample_indices = np.random.choice(len(val_dataset_full), 15, replace=False)with torch.no_grad():    for i, idx in enumerate(sample_indices):        image, true_label = val_dataset_full[idx]                # Predict        output = model(image.unsqueeze(0).to(device))        probabilities = F.softmax(output, dim=1)[0]        confidence, pred_label = probabilities.max(0)                # Convert tensors        pred_label = pred_label.item()        confidence = confidence.item() * 100        true_label = true_label.item() if isinstance(true_label, torch.Tensor) else true_label                # Plot        axes[i].imshow(image.squeeze().numpy(), cmap='gray')        axes[i].axis('off')                color = 'green' if pred_label == true_label else 'red'        axes[i].set_title(            f'True: {true_label} | Pred: {pred_label}\nConf: {confidence:.1f}%',            color=color,            fontsize=10,            fontweight='bold'        )plt.suptitle('Prediction Samples (Green=Correct, Red=Wrong)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('predictions.png', dpi=100, bbox_inches='tight')plt.show()

## 12. Confusion Matrix Analysis

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report# Get all predictions and true labelsall_preds = []all_labels = []model.eval()with torch.no_grad():    for images, labels in tqdm(val_loader, desc='Computing confusion matrix'):        images = images.to(device)        outputs = model(images)        _, predicted = outputs.max(1)                all_preds.extend(predicted.cpu().numpy())        all_labels.extend(labels.numpy())# Compute confusion matrixcm = confusion_matrix(all_labels, all_preds)# Plot confusion matrixplt.figure(figsize=(10, 8))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)plt.title('Confusion Matrix', fontsize=14, fontweight='bold')plt.ylabel('True Label')plt.xlabel('Predicted Label')plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')plt.show()# Print classification reportprint("\nClassification Report:")print(classification_report(all_labels, all_preds,                           target_names=[str(i) for i in range(10)]))

## 13. Test-Time Augmentation (TTA) for Better PredictionsApply multiple augmentations during prediction and average the results.

In [ ]:
def predict_with_tta(model, image, device, n_augmentations=5):    """Predict with test-time augmentation"""    model.eval()        # Augmentation transforms    tta_transform = transforms.Compose([        transforms.RandomRotation(10),        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),        transforms.Normalize((0.5,), (0.5,))    ])        predictions = []        with torch.no_grad():        # Original prediction        output = model(image.unsqueeze(0).to(device))        predictions.append(F.softmax(output, dim=1))                # Augmented predictions        for _ in range(n_augmentations - 1):            aug_image = tta_transform(image)            output = model(aug_image.unsqueeze(0).to(device))            predictions.append(F.softmax(output, dim=1))        # Average predictions    avg_prediction = torch.stack(predictions).mean(0)        return avg_predictionprint("Test-Time Augmentation function defined!")

## 14. Generate Test Predictions with TTA

In [ ]:
# Load test dataprint("Loading test data...")test_dataset = DigitDataset(    '/kaggle/input/digit-recognizer/test.csv',    transform=val_transform,    is_train=False)test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)# Generate predictions with TTAprint("\nGenerating predictions with Test-Time Augmentation...")predictions = []model.eval()for image in tqdm(test_loader, desc='Predicting'):    # Use TTA for better predictions    avg_output = predict_with_tta(model, image.squeeze(0), device, n_augmentations=5)    pred_label = avg_output.argmax(1).item()    predictions.append(pred_label)print(f"\nGenerated {len(predictions)} predictions")

## 15. Create Kaggle Submission File

In [ ]:
# Create submission DataFramesubmission = pd.DataFrame({    'ImageId': range(1, len(predictions) + 1),    'Label': predictions})# Save to CSVsubmission.to_csv('submission.csv', index=False)print("Submission file created successfully!")print(f"Total predictions: {len(predictions):,}")print("\nFirst 10 predictions:")print(submission.head(10))# Show prediction distributionprint("\nPrediction distribution:")print(submission['Label'].value_counts().sort_index())# Visualize prediction distributionplt.figure(figsize=(10, 5))submission['Label'].value_counts().sort_index().plot(kind='bar', color='steelblue')plt.title('Distribution of Predicted Labels', fontsize=14, fontweight='bold')plt.xlabel('Digit')plt.ylabel('Count')plt.grid(axis='y', alpha=0.3)plt.tight_layout()plt.savefig('prediction_distribution.png', dpi=100, bbox_inches='tight')plt.show()

## Summary**Key Features of This Approach:**1. **ResNet Architecture**: Uses residual blocks with skip connections for better gradient flow2. **Data Augmentation**: Random rotations, affine transforms, and perspective changes3. **Advanced Optimization**: AdamW with weight decay and learning rate scheduling4. **Test-Time Augmentation**: Multiple predictions averaged for better accuracy5. **Regularization**: Dropout, batch normalization, and weight decay6. **Monitoring**: Detailed training history, confusion matrix, and visualization**Expected Performance:**- Validation Accuracy: 99.5%+- Kaggle Score: 0.995+- Top 10% ranking**Advantages over Simple Neural Network:**- Much higher accuracy (99.5% vs 85%)- Better generalization with augmentation- Deeper architecture learns complex patterns- Robust to variations in handwriting style